# 04 — 기본값(default) 모델 성능 비교: 회귀 / 투스테이지 / ZIT

5개 트리 모델(lgbm·xgb·catboost·et·rf) + **ZIT 4종**을 **순수 라이브러리 기본값**으로 돌려 성능만 비교한다. HPO·후처리 없음.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_train_data.csv`
- **출력**: `4_output/0_baseline/default_compare/results.csv` (**34행** = reg 5 + two_stage 25 + zit 4) — §5에서 한 번에 저장
- **전처리**: 트리 공통 `PP_FIXED` + `CLIP_Y_EXTREME` + meta features — §2에서 1회, 세 모드가 같은 결과 공유 (ZIT도 die-level 행렬로 재사용)
- **모드 3종** (§4 학습 → §5 RMSE 집계):
  - **기본 회귀** (5건): die-level full-y 학습 → die→unit mean → RMSE
  - **투스테이지** (clf 5 × reg 5 = 25건): `P(Y>0) × E[Y|Y>0]` die 곱셈 → unit mean → RMSE
  - **ZIT 4종**: zero-inflated Tweedie 단일 모델 2×2 — φ(우리 Pearson / 논문충실 EQL) × bag 제약. ζ는 profile likelihood 추정. 집계는 일반 ZIT=mean·BagZIT=sum
- **파라미터**: 전 모델 라이브러리 기본값. HP 안 박음. 분류기 imbalance 보정도 OFF(=라이브러리 기본). ZIT 내부 μ/π/φ LightGBM도 라이브러리 기본값(ζ만 추정, n_em_iters=10).
- **후처리 없음**: die→unit `mean`/`sum` 집계만 (τ/position/zero_clip/집계선택 전부 미적용)
- **구성**: §1 설정 → §2 데이터·전처리 → §3 공통 헬퍼·fold → §4 학습(트리 15회 refit → ZIT 4종) → §5 결과·저장

In [1]:
import os, sys

# Google Drive 파일 ID (Colab 자동 다운로드용, 로컬은 무시)
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'   # code.zip = setup.py + utils/
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'   # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'   # preprocessing.zip
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules (zit.py φ 가드 반영본 재업로드 필요)

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/models.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리(2_preprocessing) + 모델링(3_modeling)을 패키지 접두사로 import 하도록 경로 추가
for _d in (os.path.join(PROJECT_ROOT, '2_preprocessing'), os.path.join(PROJECT_ROOT, '3_modeling')):
    if _d not in sys.path:
        sys.path.insert(0, _d)

from modules import preprocess, hpo          # 전처리 래퍼 + refit (HPO 안 씀, refit_best/refit_clf_best만)
from meta_features import add_meta_features   # die_xy / position 메타피처

# ZIT 4종 (zero-inflated Tweedie + LightGBM EM) — §4b ZIT 학습용
from modules.zit import (
    ZITboostRegressor, ZITboostEQLRegressor,
    BagZITboostRegressor, BagZITEQLRegressor,
)
from sklearn.model_selection import KFold     # unit 단위 fold (트리 refit_best와 동일 방식, ZIT refit 공용)

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'reg 모델: {hpo._models.AVAILABLE_MODELS}')
print(f'clf 모델: {hpo._models.CLF_AVAILABLE_MODELS}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
reg 모델: ['lgbm', 'xgb', 'catboost', 'et', 'rf', 'enet', 'zitboost']
clf 모델: ['lgbm', 'xgb', 'catboost', 'et', 'rf']


## 1. 설정 — 모델 / 전처리 고정값 / 기본 파라미터 (트리 + ZIT)

트리 5종 + ZIT 4종 설정을 한곳에 모은다. `PP_FIXED`·`CLIP_Y_EXTREME`은 현재 트리 노트북에 고정된 값 그대로.
기본 파라미터는 **라이브러리 기본값 + 재현(random_state)·병렬(n_jobs)·로그억제만**. HP는 일절 안 박는다.
분류기 imbalance 보정은 각 라이브러리의 '가중치 없음' 기본값을 명시로 박아 `refit_clf_best`의 자동 보정(setdefault)을 무력화한다.
ZIT는 내부 μ/π/φ LightGBM 3개를 라이브러리 기본값(`LGBM_DEFAULTS`)으로 두고, ζ(Tweedie power)만 profile likelihood로 추정(`ZETA_GRID`).

In [2]:
N_FOLDS = 5
N_JOBS  = 10   # 모델 학습 병렬도 (단독 실행이면 14로). strategy_common §8

MODELS = ['lgbm', 'xgb', 'catboost', 'et', 'rf']

CLIP_Y_EXTREME = True   # train y의 max(=1.0, 1건)를 두 번째로 큰 값으로 clip (학습 입력 안정화)

# 트리 공통 고정 전처리 (strategy_common §1)
PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

OUT_DIR = os.path.join(OUTPUT_DIR, '0_baseline', 'default_compare')
os.makedirs(OUT_DIR, exist_ok=True)


def reg_default_params(name):
    # 회귀: 라이브러리 기본값 + 재현/병렬/로그억제만 (HP 미설정)
    if name == 'lgbm':     return dict(random_state=SEED, n_jobs=N_JOBS, verbose=-1)
    if name == 'xgb':      return dict(random_state=SEED, n_jobs=N_JOBS, verbosity=0)
    if name == 'catboost': return dict(random_seed=SEED, thread_count=N_JOBS, verbose=False, allow_writing_files=False)
    if name == 'et':       return dict(random_state=SEED, n_jobs=N_JOBS)
    if name == 'rf':       return dict(random_state=SEED, n_jobs=N_JOBS)
    raise KeyError(name)


def clf_default_params(name):
    # 분류: 위와 동일 + imbalance 보정 OFF (라이브러리 '가중치 없음' 기본값을 명시 → 자동 보정 무력화)
    if name == 'lgbm':     return dict(random_state=SEED, n_jobs=N_JOBS, verbose=-1, scale_pos_weight=1.0)
    if name == 'xgb':      return dict(random_state=SEED, n_jobs=N_JOBS, verbosity=0, scale_pos_weight=1.0)
    if name == 'catboost': return dict(random_seed=SEED, thread_count=N_JOBS, verbose=False, allow_writing_files=False, auto_class_weights=None)
    if name == 'et':       return dict(random_state=SEED, n_jobs=N_JOBS, class_weight=None)
    if name == 'rf':       return dict(random_state=SEED, n_jobs=N_JOBS, class_weight=None)
    raise KeyError(name)


# --- ZIT 4종 설정 (05_zit_default와 동일 깡통 철학) ---
N_EM_ITERS = 10   # Generalized EM 반복 수 (Algorithm 1)
# ζ는 논문처럼 추정: 각 후보로 EM→train 로그우도 비교→최대 ζ* 선택 (Algorithm 2)
ZETA_GRID  = [round(z, 2) for z in np.arange(1.1, 1.85, 0.1)]   # [1.1, ..., 1.8] (8개)

# ZIT 내부 μ/π/φ LightGBM 3개 — 전부 LightGBM '라이브러리 기본값'으로 명시 (트리 깡통과 동일 철학)
LGBM_DEFAULTS = dict(
    mu_n_estimators=100, mu_learning_rate=0.1, mu_num_leaves=31, mu_max_depth=-1,
    mu_min_child_samples=20, mu_subsample=1.0, mu_colsample_bytree=1.0,
    mu_reg_alpha=0.0, mu_reg_lambda=0.0,
    pi_n_estimators=100, pi_learning_rate=0.1, pi_num_leaves=31, pi_max_depth=-1,
    pi_min_child_samples=20,
    phi_n_estimators=100, phi_learning_rate=0.1, phi_num_leaves=31, phi_max_depth=-1,
    phi_min_child_samples=20,
)

# 4종 스펙 (라벨, 클래스, bag 여부, die→unit 집계, φ 방식)
#   일반 ZIT: die가 unit health를 broadcast 학습 → mean / BagZIT: die가 unit health 몫을 분배 학습 → sum
#   phi: ζ profile은 일반 ZIT(broadcast y, 스케일 일치)에서만 돌리고 BagZIT는 같은 φ의 ζ*를 차용
ZIT_SPECS = [
    {'label': 'zit_pearson',    'cls': ZITboostRegressor,    'bag': False, 'agg': 'mean', 'phi': 'pearson'},  # zit 우리버전
    {'label': 'zit_eql',        'cls': ZITboostEQLRegressor, 'bag': False, 'agg': 'mean', 'phi': 'eql'},      # zit 논문충실
    {'label': 'bagzit_pearson', 'cls': BagZITboostRegressor, 'bag': True,  'agg': 'sum',  'phi': 'pearson'},  # bag 우리버전
    {'label': 'bagzit_eql',     'cls': BagZITEQLRegressor,   'bag': True,  'agg': 'sum',  'phi': 'eql'},      # bag 논문충실
]

print('MODELS:', MODELS)
print('N_FOLDS:', N_FOLDS, '| N_JOBS:', N_JOBS)
print('ZIT_SPECS:', [s['label'] for s in ZIT_SPECS], '| ZETA_GRID:', ZETA_GRID, '| N_EM_ITERS:', N_EM_ITERS)
print('OUT_DIR:', OUT_DIR)

MODELS: ['lgbm', 'xgb', 'catboost', 'et', 'rf']
N_FOLDS: 5 | N_JOBS: 10
ZIT_SPECS: ['zit_pearson', 'zit_eql', 'bagzit_pearson', 'bagzit_eql'] | ZETA_GRID: [np.float64(1.1), np.float64(1.2), np.float64(1.3), np.float64(1.4), np.float64(1.5), np.float64(1.6), np.float64(1.7), np.float64(1.8)] | N_EM_ITERS: 10
OUT_DIR: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\0_baseline\default_compare


## 2. 데이터 로드 + 고정 전처리 (1회) — 트리·ZIT 공유

`preprocess.run`(cleaning + spatial imputation + outlier) → meta features. 전 모델·전 모드가 같은 전처리 결과를 공유한다.
ZIT는 die-level로 학습하므로, 같은 전처리 결과를 `X_train/val/test`(np.float64 행렬) + die→unit 매핑(`uid_*_die`) + unit health를 4 die에 broadcast한 `y_train_die`로도 만들어 둔다.

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

# train y 극단값(1.0 1건)만 두 번째로 큰 값으로 clip
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = int((y_raw >= 1.0).sum())
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 -> {second_max:.6f} clip, {n_clipped}개 샘플')

# 고정 전처리 1회
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

# 메타피처 (트리: position raw 정수 + die_x/die_y 연속형)
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

# unit-level 정답 (index=ufs_serial)
y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit_s   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit_s  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

# --- ZIT용 die-level 행렬/매핑/타깃 (같은 전처리 결과 재사용 — 결측 채워져 NaN 없음) ---
X_train = xs_train[feat_cols_clean].values.astype(np.float64)
X_val   = xs_val[feat_cols_clean].values.astype(np.float64)
X_test  = xs_test[feat_cols_clean].values.astype(np.float64)

uid_train_die = xs_train[KEY_COL].values
uid_val_die   = xs_val[KEY_COL].values
uid_test_die  = xs_test[KEY_COL].values

# die-level 정답: 각 die에 자기 unit health를 broadcast (full-y)
y_train_die = xs_train[KEY_COL].map(y_train_unit_s).values.astype(np.float64)
assert not np.isnan(y_train_die).any(), 'y_train_die NaN — train die의 unit이 y에 없음'

print(f'[전처리 완료] feat_cols: {len(feat_cols_clean)}')
print(f'  xs_train: {xs_train.shape}, xs_val: {xs_val.shape}, xs_test: {xs_test.shape}')
print(f'  X_train: {X_train.shape} (NaN={int(np.isnan(X_train).sum())}), X_val: {X_val.shape}, X_test: {X_test.shape}')
print(f'  unit train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}, test={len(y_test_unit_s):,}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 -> 0.097417 clip, 1개 샘플
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1031 (56개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1031
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 926개
    컬럼: 1031 → 926 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=30%
  제거: 5개, 잔여: 921개
    컬럼: 926 → 921 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 894개
    컬럼: 921 → 894 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 330개, 잔여: 564개
    컬럼: 894 → 564 (330개 제거)
    DataFrame: (104748, 624)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, 

## 3. 공통 헬퍼 · fold (세 모드 공유)

die→unit 집계·unit RMSE·split별 매핑(UID)·정답(YK)·unit 단위 fold(FOLDS)·ZIT 모델 빌더를 한곳에 정의해 §4·§5가 공유한다.
- 집계: 기본 회귀/투스테이지/일반 ZIT = **mean**, BagZIT = **sum** (die가 unit health 몫을 분배 학습하므로)
- fold: `refit_best`(트리 내부)와 동일하게 **unit ID 단위** KFold(shuffle + SEED) → 트리·ZIT OOF가 같은 분할 위에서 나옴 (같은 unit의 4 die가 train/val에 섞이면 leakage)

In [4]:
# --- die→unit 집계 (how='mean': 회귀/투스테이지/일반 ZIT, how='sum': BagZIT) ---
def unit_agg(uid_die, die_pred, how='mean'):
    g = pd.DataFrame({KEY_COL: uid_die, 'v': np.asarray(die_pred)}).groupby(KEY_COL, sort=False)['v']
    return g.mean() if how == 'mean' else g.sum()

# 예측 Series를 정답 index 순서에 맞춰 unit RMSE
def rmse_unit(pred_s, y_s):
    p = pred_s.loc[y_s.index]
    return float(np.sqrt(np.mean((p.values - y_s.values) ** 2)))

# split별 die→unit 매핑 key + 정답 (reg/two_stage die 예측 집계용)
UID = {'oof': xs_train[KEY_COL].values, 'val': xs_val[KEY_COL].values, 'test': xs_test[KEY_COL].values}
YK  = {'oof': y_train_unit_s, 'val': y_val_unit_s, 'test': y_test_unit_s}

# unit ID 단위 KFold — refit_best(_make_unit_folds)와 동일 방식(ufs_serial 등장순서 + SEED) → 트리·ZIT OOF가 같은 분할
unique_units = y_train_unit_s.index.values
FOLDS = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(unique_units))

# --- ZIT 모델 빌더/적합 헬퍼 ---
def make_zit(spec, zeta):
    return spec['cls'](
        zeta=zeta, n_em_iters=N_EM_ITERS,
        random_state=SEED, n_jobs=N_JOBS, verbose=-1, device='cpu', **LGBM_DEFAULTS,
    )

def fit_zit(model, Xtr, ytr, uid_tr, is_bag):
    if is_bag:
        model.fit(Xtr, ytr, unit_id=uid_tr)   # BagZIT: unit_id 필수
    else:
        model.fit(Xtr, ytr)
    return model

print(f'[helpers] FOLDS={len(FOLDS)} (unit 단위) · UID/YK splits={list(UID)} · unit_agg/rmse_unit/make_zit/fit_zit 준비')

[helpers] FOLDS=5 (unit 단위) · UID/YK splits=['oof', 'val', 'test'] · unit_agg/rmse_unit/make_zit/fit_zit 준비


## 4. 모델 학습 — (4a) 트리 15회 refit → (4b) ZIT 4종

(4a) 트리: 모델 5개 × {기본회귀(full y) · 분류 P(Y>0) · 투스테이지 회귀 E[Y|Y>0]} = **15회** 5-fold refit → die 예측 `cache`.
(4b) ZIT: 2×2 스펙을 ζ profile(일반 ZIT) → ζ* 차용(BagZIT) → 5-fold refit → unit 예측 `zit_cache`.
die 예측을 캐싱해 두면 투스테이지 25조합·ZIT 4종 RMSE를 §5에서 재학습 없이 집계만으로 만든다.

> RF/ExtraTrees 회귀는 라이브러리 기본값 `max_features=1.0`(전 피처)이라 다소 느릴 수 있음 — 의도된 기본값.
> 참고 시간: 트리 refit은 모델별 편차 큼(catboost/et/rf가 김), ZIT 4종은 일반 ZIT의 ζ profile(8 후보) 포함 대략 1~1.5h.

In [5]:
# (4a) 트리 15회 refit — 모델 5개 × {기본회귀(full y) · 분류 P(Y>0) · 투스테이지 회귀 E[Y|Y>0]=y>0 only}
#   die-level 예측을 cache에 저장 → 투스테이지 25조합은 §5에서 재학습 없이 곱셈만으로 만든다.
cache = {}   # cache[name] = {'basic':{split:die}, 'clf':{split:die}, 'tsreg':{split:die}}
t_all = time.time()

for name in MODELS:
    t0 = time.time()
    print(f'\n===== {name} =====')

    # (1) 기본 회귀 — full y, die broadcast
    basic = hpo.refit_best(
        xs_train, xs_val, xs_test, ys_input['train'], feat_cols_clean,
        model_name=name, best_params=reg_default_params(name),
        n_folds=N_FOLDS, already_resolved=True, n_jobs=None,
    )
    # (2) 투스테이지 분류 — P(Y>0)
    clf = hpo.refit_clf_best(
        xs_train, xs_val, xs_test, ys_input['train'], feat_cols_clean,
        model_name=name, best_params=clf_default_params(name),
        n_folds=N_FOLDS, already_resolved=True, n_jobs=None,
    )
    # (3) 투스테이지 회귀 — E[Y|Y>0], y>0 only
    tsreg = hpo.refit_best(
        xs_train, xs_val, xs_test, ys_input['train'], feat_cols_clean,
        model_name=name, best_params=reg_default_params(name),
        n_folds=N_FOLDS, y_positive_only=True, already_resolved=True, n_jobs=None,
    )

    cache[name] = {
        'basic': {'oof': basic['oof_pred_die'], 'val': basic['val_pred_die'], 'test': basic['test_pred_die']},
        'clf':   {'oof': clf['oof_proba_die'],  'val': clf['val_proba_die'],  'test': clf['test_proba_die']},
        'tsreg': {'oof': tsreg['oof_pred_die'], 'val': tsreg['val_pred_die'], 'test': tsreg['test_pred_die']},
    }
    print(f'[{name}] done ({time.time()-t0:.0f}s)')

print(f'\n[트리 refit 완료] {time.time()-t_all:.0f}s')


===== lgbm =====
[refit fold 1/5] tr_units=20949, vl_units=5238
[refit fold 2/5] tr_units=20949, vl_units=5238
[refit fold 3/5] tr_units=20950, vl_units=5237
[refit fold 4/5] tr_units=20950, vl_units=5237
[refit fold 5/5] tr_units=20950, vl_units=5237
[clf refit fold 1/5] tr_units=20949, vl_units=5238, pos_ratio=0.290
[clf refit fold 2/5] tr_units=20949, vl_units=5238, pos_ratio=0.294
[clf refit fold 3/5] tr_units=20950, vl_units=5237, pos_ratio=0.293
[clf refit fold 4/5] tr_units=20950, vl_units=5237, pos_ratio=0.292
[clf refit fold 5/5] tr_units=20950, vl_units=5237, pos_ratio=0.291
[refit fold 1/5] tr_units=20949, vl_units=5238
[refit fold 2/5] tr_units=20949, vl_units=5238
[refit fold 3/5] tr_units=20950, vl_units=5237
[refit fold 4/5] tr_units=20950, vl_units=5237
[refit fold 5/5] tr_units=20950, vl_units=5237
[lgbm] done (58s)

===== xgb =====
[refit fold 1/5] tr_units=20949, vl_units=5238
[refit fold 2/5] tr_units=20949, vl_units=5238
[refit fold 3/5] tr_units=20950, vl_units=5

In [6]:
# (4b) ZIT 4종 fit — 일반 ZIT는 ζ profile likelihood로 ζ* 결정, BagZIT는 같은 φ의 ζ* 차용 → 5-fold refit
zit_cache = {}      # label -> {'zeta_star', 'oof', 'val', 'test'(unit-level Series)}
zeta_by_phi = {}    # phi -> ζ* : 일반 ZIT가 profile로 결정, 같은 φ의 BagZIT가 차용
t_all = time.time()

for spec in ZIT_SPECS:
    label, is_bag, agg, phi = spec['label'], spec['bag'], spec['agg'], spec['phi']
    print(f'\n===== {label} (bag={is_bag}, agg={agg}, phi={phi}) =====')

    # (1) ζ 결정
    #   - 일반 ZIT: profile likelihood — die가 broadcast y를 직접 학습하므로 score_loglik 스케일 일치
    #   - BagZIT  : 같은 φ의 일반 ZIT ζ* 차용 — BagZIT은 die가 unit의 ¼ 몫을 학습(μ≈unit/4)해
    #               broadcast-y profile이 스케일 불일치 → 부정확한 profile 대신 일치하는 일반 ZIT ζ* 사용
    if not is_bag:
        t0 = time.time()
        ll_by_zeta = {}
        for z in ZETA_GRID:
            m = fit_zit(make_zit(spec, z), X_train, y_train_die, uid_train_die, is_bag)
            ll_by_zeta[z] = m.score_loglik(X_train, y_train_die)
        zeta_star = max(ll_by_zeta, key=ll_by_zeta.get)
        zeta_by_phi[phi] = zeta_star            # 같은 φ의 BagZIT가 차용
        print(f'  [ζ*] {zeta_star:.2f} (profile, train loglik 최대) | {time.time()-t0:.0f}s')
    else:
        zeta_star = zeta_by_phi[phi]            # 대응 일반 ZIT(zit_{phi})의 ζ* 차용
        print(f'  [ζ*] {zeta_star:.2f} (zit_{phi} ζ* 차용)')

    # (2) ζ* 5-fold refit — die 예측 (1-π)μ → oof/val/test (val/test는 fold 평균)
    oof_die  = np.full(len(X_train), np.nan)
    val_die  = np.zeros(len(X_val))
    test_die = np.zeros(len(X_test))
    t1 = time.time()
    for tr_uidx, vl_uidx in FOLDS:
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask  = np.isin(uid_train_die, tr_units)   # unit mask → die mask
        vl_mask  = np.isin(uid_train_die, vl_units)

        model = fit_zit(make_zit(spec, zeta_star),
                        X_train[tr_mask], y_train_die[tr_mask], uid_train_die[tr_mask], is_bag)
        oof_die[vl_mask] = model.predict(X_train[vl_mask])   # (1-π)μ, 음수 clip은 predict 내부
        val_die  += model.predict(X_val)  / N_FOLDS
        test_die += model.predict(X_test) / N_FOLDS
    assert not np.isnan(oof_die).any(), f'{label}: OOF die 미커버 — fold 누락'
    print(f'  [refit] {time.time()-t1:.0f}s')

    zit_cache[label] = {
        'zeta_star': zeta_star,
        'oof':  unit_agg(uid_train_die, oof_die,  agg),
        'val':  unit_agg(uid_val_die,   val_die,  agg),
        'test': unit_agg(uid_test_die,  test_die, agg),
    }

print(f'\n[ZIT 4종 완료] {time.time()-t_all:.0f}s')


===== zit_pearson (bag=False, agg=mean, phi=pearson) =====
  [ζ*] 1.80 (profile, train loglik 최대) | 1573s
  [refit] 998s

===== zit_eql (bag=False, agg=mean, phi=eql) =====
  [ζ*] 1.10 (profile, train loglik 최대) | 1612s
  [refit] 1007s

===== bagzit_pearson (bag=True, agg=sum, phi=pearson) =====
  [ζ*] 1.80 (zit_pearson ζ* 차용)
  [refit] 1013s

===== bagzit_eql (bag=True, agg=sum, phi=eql) =====
  [ζ*] 1.10 (zit_eql ζ* 차용)
  [refit] 1028s

[ZIT 4종 완료] 7231s


## 5. 결과 비교 + 저장 — results.csv (34행)

세 모드의 die/unit 예측(`cache`·`zit_cache`)을 unit RMSE로 집계해 한 표로 합치고 **한 번만 저장**한다.
- 기본 회귀 5건 · 투스테이지 25건(clf 5 × reg 5, die 곱 → unit mean) · ZIT 4건 = **34행**
- 후처리 없음 (τ/position/zero_clip/집계선택 미적용)

In [7]:
# 기본 회귀 RMSE (5건) — die 예측 → unit mean → RMSE
rows = []
for name in MODELS:
    die = cache[name]['basic']
    r = {sp: rmse_unit(unit_agg(UID[sp], die[sp], 'mean'), YK[sp]) for sp in ('oof', 'val', 'test')}
    rows.append({'mode': 'reg', 'clf': '-', 'reg': name,
                 'oof_rmse': r['oof'], 'val_rmse': r['val'], 'test_rmse': r['test']})

reg_df = pd.DataFrame(rows).sort_values('val_rmse').reset_index(drop=True)
print('=== 기본 회귀 (val 오름차순) ===')
print(reg_df.to_string(index=False, float_format='%.6f'))

=== 기본 회귀 (val 오름차순) ===
mode clf      reg  oof_rmse  val_rmse  test_rmse
 reg   - catboost  0.005540  0.005733   0.008432
 reg   -     lgbm  0.005540  0.005735   0.008434
 reg   -      xgb  0.005576  0.005741   0.008434
 reg   -       et  0.005577  0.005762   0.008446
 reg   -       rf  0.005589  0.005770   0.008452


In [8]:
# 투스테이지 그리드 RMSE (clf 5 × reg 5 = 25건) — die-level P(Y>0) × E[Y|Y>0] 곱 → unit mean → RMSE (후처리 없음)
grid_rows = []
for c in MODELS:                 # 분류기
    proba = cache[c]['clf']
    for r in MODELS:             # 회귀기
        regd = cache[r]['tsreg']
        res = {}
        for sp in ('oof', 'val', 'test'):
            final_die = np.asarray(proba[sp]) * np.asarray(regd[sp])   # die 단위 곱
            res[sp] = rmse_unit(unit_agg(UID[sp], final_die, 'mean'), YK[sp])
        grid_rows.append({'mode': 'two_stage', 'clf': c, 'reg': r,
                          'oof_rmse': res['oof'], 'val_rmse': res['val'], 'test_rmse': res['test']})

ts_df = pd.DataFrame(grid_rows).sort_values('val_rmse').reset_index(drop=True)
print('=== 투스테이지 그리드 (val 오름차순) ===')
print(ts_df.to_string(index=False, float_format='%.6f'))

=== 투스테이지 그리드 (val 오름차순) ===
     mode      clf      reg  oof_rmse  val_rmse  test_rmse
two_stage     lgbm     lgbm  0.005523  0.005718   0.008418
two_stage      xgb     lgbm  0.005539  0.005719   0.008413
two_stage catboost     lgbm  0.005527  0.005720   0.008417
two_stage     lgbm catboost  0.005526  0.005721   0.008419
two_stage     lgbm      xgb  0.005543  0.005722   0.008421
two_stage      xgb catboost  0.005541  0.005722   0.008413
two_stage      xgb      xgb  0.005559  0.005722   0.008416
two_stage catboost catboost  0.005530  0.005722   0.008417
two_stage catboost      xgb  0.005547  0.005723   0.008420
two_stage     lgbm       rf  0.005526  0.005724   0.008417
two_stage     lgbm       et  0.005525  0.005724   0.008420
two_stage      xgb       rf  0.005542  0.005725   0.008411
two_stage catboost       rf  0.005530  0.005725   0.008416
two_stage      xgb       et  0.005542  0.005726   0.008415
two_stage catboost       et  0.005530  0.005727   0.008419
two_stage       rf     lgbm

In [9]:
# ZIT 4종 RMSE — die 예측은 §4b에서 이미 unit 집계됨(일반=mean/Bag=sum)
zit_rows = []
for spec in ZIT_SPECS:
    label = spec['label']
    c = zit_cache[label]
    zit_rows.append({
        'mode': 'zit', 'clf': '-',
        'reg': f"{label}_z{c['zeta_star']:.2f}",   # ζ*는 reg 라벨에 기록
        'oof_rmse':  rmse_unit(c['oof'],  y_train_unit_s),
        'val_rmse':  rmse_unit(c['val'],  y_val_unit_s),
        'test_rmse': rmse_unit(c['test'], y_test_unit_s),
    })

zit_df = pd.DataFrame(zit_rows).sort_values('val_rmse').reset_index(drop=True)
print('=== ZIT 4종 깡통 (val 오름차순) ===')
print(zit_df.to_string(index=False, float_format='%.6f'))

# reg(5) + two_stage(25) + zit(4) = 34행 통합 저장 (단일 저장)
results = pd.concat([reg_df, ts_df, zit_df], ignore_index=True)
out_path = os.path.join(OUT_DIR, 'results.csv')
results.to_csv(out_path, index=False)
print(f'\n저장: {out_path}  ({len(results)}행)')

print('\n=== 전체 통합 (val 오름차순, head 15) ===')
print(results.sort_values('val_rmse').head(15).to_string(index=False, float_format='%.6f'))

=== ZIT 4종 깡통 (val 오름차순) ===
mode clf                  reg  oof_rmse  val_rmse  test_rmse
 zit   - bagzit_pearson_z1.80  0.005544  0.005715   0.008418
 zit   -    zit_pearson_z1.80  0.005532  0.005715   0.008413
 zit   -        zit_eql_z1.10  0.005515  0.005722   0.008424
 zit   -     bagzit_eql_z1.10  0.005531  0.005730   0.008430

저장: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\0_baseline\default_compare\results.csv  (34행)

=== 전체 통합 (val 오름차순, head 15) ===
     mode      clf                  reg  oof_rmse  val_rmse  test_rmse
      zit        - bagzit_pearson_z1.80  0.005544  0.005715   0.008418
      zit        -    zit_pearson_z1.80  0.005532  0.005715   0.008413
two_stage     lgbm                 lgbm  0.005523  0.005718   0.008418
two_stage      xgb                 lgbm  0.005539  0.005719   0.008413
two_stage catboost                 lgbm  0.005527  0.005720   0.008417
two_stage     lgbm             catboost  0.005526  0.005721   0.008419
two_stage     lgbm                  xgb